# Where should care navigators start? Exploring CMS hospital readmissions

Every number comes from the dbt marts in `out/heally.duckdb`, built from two CMS Provider Data files ([HRRP 9n3s-kdb3](https://data.cms.gov/provider-data/dataset/9n3s-kdb3), [Hospital General Information xubh-q36u](https://data.cms.gov/provider-data/dataset/xubh-q36u)). Run `make run` first.

In [1]:
import duckdb, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
con = duckdb.connect('out/heally.duckdb', read_only=True)
q = lambda s: con.execute(s).fetchdf()
pd.set_option('display.width', 160)

## 1. The headline numbers
HRRP covers a three-year window, so the business case is annualized. Cost per readmission and the avertable share are **assumptions** set in `dbt_project.yml`.

In [2]:
q('select * from marts.mart_business_case').T

,0
performance_period_years,3.000000e+00
worklist_pairs,4.316000e+03
tier_a_pairs,4.570000e+02
tier_a_share_of_pairs,1.059000e-01
excess_readmissions_period,1.224530e+04
excess_readmissions_per_year,4.081800e+03
tier_a_share_of_excess,5.002000e-01
tier_a_excess_per_1000_discharges,1.990000e+01
all_pairs_excess_per_1000_discharges,1.080000e+01
assumed_avertable_share,1.000000e-01


## 2. How concentrated is the problem?
If outreach followed the ranking, how much of the excess is covered by the first N hospital-condition pairs?

In [3]:
w = q('select outreach_rank, cumulative_share from marts.mart_navigator_worklist order by 1')
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(w.outreach_rank / len(w) * 100, w.cumulative_share * 100)
ax.axhline(50, ls='--', c='grey'); ax.set_xlabel('% of hospital-condition pairs worked (largest first)'); ax.set_ylabel('% of excess readmissions covered')
ax.set_title('Half of the excess sits in about 11% of pairs'); plt.tight_layout(); plt.savefig('docs/img/concentration.png', dpi=160); plt.show()

## 3. Which conditions drive it?

In [4]:
q("""select condition_label, count(*) pairs, round(sum(excess_readmissions)) excess, round(1000*sum(excess_readmissions)/sum(discharges),1) per_1000_discharges
from marts.mart_navigator_worklist group by 1 order by excess desc""")

,condition_label,pairs,excess,per_1000_discharges
0,Heart failure,1187,4856.0,12.1
1,Pneumonia,1189,4526.0,11.0
2,Heart attack,703,1130.0,8.4
3,Chronic obstructive pulmonary disease,880,1072.0,8.8
4,Coronary artery bypass graft surgery,221,341.0,10.3
5,Hip or knee replacement,136,321.0,9.1


## 4. Does the CMS star rating line up with readmission performance?
A useful sanity check on the data: worse-rated hospitals should readmit more than expected.

In [5]:
s = q('select overall_star_rating, count(*) hospitals, round(avg(avg_excess_readmission_ratio),4) avg_ratio, round(sum(excess_readmissions)) excess from marts.mart_hospital_scorecard group by 1 order by 1')
s

,overall_star_rating,hospitals,avg_ratio,excess
0,1,170,1.0454,1194.0
1,2,569,1.0230,3631.0
2,3,846,1.0056,3967.0
3,4,779,0.9859,2534.0
4,5,290,0.9642,818.0
5,<NA>,401,0.9723,102.0


In [6]:
t = s.dropna(); fig, ax = plt.subplots(figsize=(5, 3)); ax.bar(t.overall_star_rating.astype(int).astype(str), t.avg_ratio - 1)
ax.axhline(0, c='k', lw=.6); ax.set_xlabel('CMS overall star rating'); ax.set_ylabel('avg excess ratio minus 1'); plt.tight_layout(); plt.savefig('docs/img/stars.png', dpi=160); plt.show()

## 5. Where are partners most exposed? (state view for health plan and employer reporting)

In [7]:
q('select state, hospitals, excess_readmissions, excess_readmissions_per_1000_discharges per_1000, share_pairs_above_expected from marts.mart_state_exposure order by excess_readmissions desc limit 10')

,state,hospitals,excess_readmissions,per_1000,share_pairs_above_expected
0,FL,159,1421.55,7.21,0.6237
1,CA,224,1278.61,6.72,0.6094
2,NY,119,871.02,6.19,0.5857
3,MA,52,812.59,8.80,0.6800
4,NJ,61,707.51,8.52,0.6892
5,IL,103,704.89,6.06,0.6179
6,TX,191,678.14,4.31,0.5675
7,PA,114,637.35,5.92,0.5588
8,OH,104,479.70,5.28,0.5503
9,MI,74,436.44,6.44,0.5244


## 6. What the data cannot tell us
- HRRP covers **Medicare fee-for-service** patients, mostly 65 and over, not every Wellthy member.
- Excess readmissions are **estimated** from published rates: discharges x (predicted - expected) / 100. CMS suppresses small cells, so only 8,037 of 18,330 hospital-condition rows are computable.
- 20 hospitals appear in HRRP but not in Hospital General Information (kept and flagged, see INC-3).

In [8]:
q('''select count(*) filter (where excess_readmissions is not null) computable, count(*) total, count(*) filter (where not exists (select 1 from marts.dim_hospital d where d.facility_id=f.facility_id and d.in_general_info)) from_unknown_hospitals
from marts.fct_hospital_readmissions f''')

,computable,total,from_unknown_hospitals
0,8037,18330,120
